In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/kaggle/input/predict-student-performance-dataset/data.csv")

In [ ]:
df.head(10)

In [ ]:
# Separate features
X = df.iloc[:, df.columns != 'Grades']
y = df['Grades']

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Scale the features
scaler = StandardScaler()

# Fit the scaler to the training data and transform the features
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data based on the scaler fitted on the training data
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=50),
    'XGBoost': XGBRegressor(random_state=50),
    'SVR': SVR(kernel='rbf')
}

In [ ]:
models

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    import numpy as np
    
    # Calculate evaluation metrics
    mse = mean_squared_error(y_true, y_pred)
    metrics = {
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }
    
    # Print performance metrics
    print(f"\n{model_name} Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.2f}")
    
    # Return the metrics as a dictionary
    return metrics

In [ ]:
# Train and evaluate models
results = {}

for model_name, model_instance in models.items():
    print(f"\nTraining {model_name}...")
    
    # Train the model
    model_instance.fit(X_train_scaled, y_train)
    
    # Generate predictions
    predictions = model_instance.predict(X_test_scaled)
    
    # Evaluate and store the results
    metrics = evaluate_model(y_test, predictions, model_name)
    results[model_name] = metrics

In [ ]:
import matplotlib.pyplot as plt

# Extract model names and R2 scores from results
model_names = list(results.keys())
r2_scores = [metrics['R2'] for metrics in results.values()]

# Create the plot
plt.figure(figsize=(12, 6))
plt.bar(model_names, r2_scores, color='green', edgecolor='black')

# Customize the plot
plt.title('Model Comparison - R² Scores', fontsize=16)
plt.xlabel('Models', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
import numpy as np
best_model_name = model_names[np.argmax(r2_scores)]
best_model = models[best_model_name]

In [ ]:
best_model